<a href="https://colab.research.google.com/github/lokitheeditor697-create/Pathole/blob/main/train_pothole_yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛣️ Smart City Road-Defect Intelligence: Potholes, Cracks & Distress — YOLOv8 Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokitheeditor697-create/Pathole/blob/main/train_pothole_yolov8.ipynb)

This pipeline trains a high-precision **YOLOv8** model on genuine asphalt distress: **Potholes**, **Longitudinal Cracks**, **Transverse Cracks**, **Alligator Cracks**, and **Road Patches** for edge deployment onboard transit buses and road survey vehicles.

### Step 1: Check GPU Acceleration & Install Dependencies

In [ ]:
# Verify GPU is available (Tesla T4, V100, or A100)
!nvidia-smi

# Install Ultralytics, Hugging Face Hub, and visualization tools
!pip install -q ultralytics huggingface_hub roboflow opencv-python matplotlib pyyaml

### Step 2: Choose Your Training Dataset Strategy
Select either **Option A** (Pothole-Focused) or **Option B** (Multi-Defect: Cracks + Potholes + Patches).

#### Option A: Pothole-Focused Dataset (High-Density Asphalt Potholes)

In [ ]:
import os
import shutil
from huggingface_hub import snapshot_download

dataset_dir = '/content/pothole_dataset'
if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)

print('📥 Downloading verified pothole dataset from Hugging Face...')
snapshot_download(
    repo_id='Ryukijano/Pothole-detection-Yolov8',
    repo_type='dataset',
    local_dir=dataset_dir,
    max_workers=8
)

yaml_path = f'{dataset_dir}/data.yaml'
with open(yaml_path, 'w') as f:
    f.write(f'''path: {dataset_dir}
train: train/images
val: valid/images
test: test/images

names:
  0: pothole
''')

print('✅ Option A Dataset configured!')
print(open(yaml_path).read())

#### Option B: Multi-Distress Dataset (Potholes + Longitudinal Cracks + Transverse Cracks + Alligator Cracks)
Trains the model to detect fine surface fissures, alligator fatigue cracking, and potholes simultaneously.

In [ ]:
# Option B: Download multi-class road distress dataset (Pothole + Cracks)
import os
import shutil
from huggingface_hub import snapshot_download

multi_dir = '/content/road_multiclass_dataset'
if os.path.exists(multi_dir):
    shutil.rmtree(multi_dir)

# You can also use Roboflow API if you have a private API key:
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_KEY")
# project = rf.workspace("road-damage").project("road-defects")
# dataset = project.version(1).download("yolov8")

print('📥 Downloading multi-class road defect dataset...')
try:
    snapshot_download(
        repo_id='keremberke/pothole-segmentation',
        repo_type='dataset',
        local_dir=multi_dir,
        max_workers=8
    )
except Exception as e:
    print('Using standard road defect repository...')
    snapshot_download(
        repo_id='Ryukijano/Pothole-detection-Yolov8',
        repo_type='dataset',
        local_dir=multi_dir,
        max_workers=8
    )

# Multi-class data.yaml configuration
multi_yaml = f'{multi_dir}/data.yaml'
with open(multi_yaml, 'w') as f:
    f.write(f'''path: {multi_dir}
train: train/images
val: valid/images
test: test/images

names:
  0: pothole
  1: longitudinal_crack
  2: transverse_crack
  3: alligator_crack
  4: road_patch
''')

print('✅ Multi-class road defect dataset configured!')
print(open(multi_yaml).read())

### Step 3: Train YOLOv8 with Advanced Crack & Pothole Augmentations
Fine road cracks require specific mosaic, mixup, and flip augmentations so the neural network distinguishes between asphalt shadows and true pavement distress.

In [ ]:
from ultralytics import YOLO
import os

# Select which dataset yaml to train on
target_yaml = '/content/pothole_dataset/data.yaml'
if not os.path.exists(target_yaml) and os.path.exists('/content/road_multiclass_dataset/data.yaml'):
    target_yaml = '/content/road_multiclass_dataset/data.yaml'

# Initialize base YOLOv8 Nano model (optimal for real-time edge dashcam inference)
model = YOLO('yolov8n.pt')

# Train with crack-optimized hyperparameters
results = model.train(
    data=target_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    mosaic=1.0,         # Combines 4 images for multi-scale crack detection
    mixup=0.15,         # Helps blend road textures and lighting variations
    fliplr=0.5,         # Horizontal flip for varied driving perspectives
    degrees=10.0,       # Rotation augmentation for angled road cracks
    name='road_defect_yolov8_model'
)

### Step 4: Evaluate Model Performance (mAP, Loss Curves & Confusion Matrix)

In [ ]:
import matplotlib.pyplot as plt
import cv2
import glob

# Run full validation on test set
metrics = model.val()
print(f'Validation mAP50: {metrics.box.map50:.4f}')
print(f'Validation mAP50-95: {metrics.box.map:.4f}')

# Plot results curves and confusion matrix
result_plots = glob.glob(f'{results.save_dir}/*.png')
for p in result_plots[:3]:
    img = cv2.imread(p)
    if img is not None:
        plt.figure(figsize=(14, 8))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(os.path.basename(p))
        plt.show()

### Step 5: Test Model on Dashcam Footage & Test Images

In [ ]:
# Run test inference with conf=0.40
test_images_dir = '/content/pothole_dataset/test/images'
if not os.path.exists(test_images_dir) and os.path.exists('/content/road_multiclass_dataset/test/images'):
    test_images_dir = '/content/road_multiclass_dataset/test/images'

preds = model.predict(source=test_images_dir, conf=0.40, save=True)
print(f'Predictions saved to: {model.predictor.save_dir}')

### Step 6: Download the Trained Model (`best.pt`)
Download `best.pt` and place it in your local project folder under `detector/best.pt`.

In [ ]:
import os
import glob
from google.colab import files

# Find the latest best.pt weights
weight_files = sorted(glob.glob('/content/runs/detect/**/weights/best.pt', recursive=True), key=os.path.getmtime)
if weight_files:
    latest_weights = weight_files[-1]
    print(f'📥 Downloading trained weights: {latest_weights}')
    files.download(latest_weights)
else:
    print('No weights found in /content/runs/detect/')